# Causal Language Modeling


In [1]:
import sys
from pathlib import Path

ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'course_tools').exists() or (path / 'picollm').exists()), Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pathlib import Path
import torch
from course_tools import CharTokenizer, TinyConfig, TinyTransformerLM

LECTURE_NOTE_TITLE = 'Causal Language Modeling'
print(f'Lecture note: {LECTURE_NOTE_TITLE}')


Lecture note: Causal Language Modeling


## Next-token targets are the input shifted by one


In [2]:
text = 'causal modeling'
tokenizer = CharTokenizer.build([text])
ids = torch.tensor(tokenizer.encode(text, add_bos=True, add_eos=True))
x = ids[:-1]
y = ids[1:]
print('x:', x)
print('y:', y)
print('decoded x:', tokenizer.decode(x.tolist()))
print('decoded y:', tokenizer.decode(y.tolist()))


x: tensor([ 1,  8,  7, 18, 17,  7, 13,  6, 14, 16,  9, 10, 13, 12, 15, 11])
y: tensor([ 8,  7, 18, 17,  7, 13,  6, 14, 16,  9, 10, 13, 12, 15, 11,  2])
decoded x: causal modeling
decoded y: causal modeling


## A decoder-only model produces logits for every position


In [3]:
model = TinyTransformerLM(TinyConfig(vocab_size=len(tokenizer.stoi), block_size=32, d_model=32, n_heads=4, n_layers=1, d_ff=64))
logits, loss, _ = model(x[None, :], targets=y[None, :])
print('logits shape:', logits.shape)
print('loss:', loss.item())


logits shape: torch.Size([1, 16, 19])
loss: 22.518695831298828


## Each position predicts the next token, not itself


In [4]:
top_ids = torch.argmax(logits[0], dim=-1)
print('predicted ids:', top_ids)
print('target ids   :', y)


predicted ids: tensor([ 1,  8,  7, 18, 17,  7, 13,  6, 14, 16,  9, 10, 13, 12, 15, 11])
target ids   : tensor([ 8,  7, 18, 17,  7, 13,  6, 14, 16,  9, 10, 13, 12, 15, 11,  2])


## Teacher forcing scores all positions in parallel


During training we already know the full target sequence, so we compute loss for every position at once.
During generation we only know the prefix, so decoding becomes sequential.


## External reference repos


**RASBT**
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/01_main-chapter-code/gpt_train.py
**NANOCHAT**
- https://github.com/karpathy/nanochat/blob/master/nanochat/dataloader.py
- https://github.com/karpathy/nanochat/blob/master/scripts/base_train.py
